In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.futbol.silver_players;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.futbol.silver_players (
    match_id STRING COMMENT 'ID único del partido extraído de la URL',
    team_name STRING COMMENT 'Nombre del equipo',
    player_name STRING COMMENT 'Nombre del jugador',
    player_position STRING COMMENT 'Posición del jugador'
)
USING DELTA
COMMENT 'Tabla Silver de los jugadores de los partidos de fútbol'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'silver'
)

In [0]:
%sql
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT match_id) AS total_matchs,
    COUNT(DISTINCT team_name) AS total_teams,
    COUNT(DISTINCT player_position) AS total_positions
FROM
    workspace.futbol.silver_players
;

In [0]:
%sql
MERGE INTO workspace.futbol.silver_players AS target
USING (
    SELECT
        match_id,
        team_name,
        player_name,
        player_position
    FROM (
        SELECT
            REPLACE(substring_index(parsed_match[0].url, '-', -1), '/', '') AS match_id,
            DECODE(ENCODE(team.name, 'ISO-8859-1'), 'UTF-8') AS team_name,
            DECODE(ENCODE(team_players.name, 'ISO-8859-1'), 'UTF-8') AS player_name,
            DECODE(ENCODE(team_players.roleName, 'ISO-8859-1'), 'UTF-8') AS player_position,
            ROW_NUMBER() OVER (PARTITION BY REPLACE(substring_index(parsed_match[0].url, '-', -1), '/', ''), team.name, team_players.name ORDER BY team_players.roleName) AS rn
        FROM (
            SELECT from_json(match, 'array<struct<`@context`:string, `@type`:string, url:string, `@graph`:array<struct<`@type`:string, name:string, athlete:array<struct<`@type`:string, name:string, roleName:string>>>>>>') AS parsed_match
            FROM workspace.futbol.bronze_matchs
        )
            LATERAL VIEW EXPLODE(parsed_match[1].`@graph`) t AS team
            LATERAL VIEW EXPLODE(team.athlete) p AS team_players
        WHERE
            team.`@type` = 'SportsTeam'
    )
    WHERE rn = 1
) AS source
ON target.match_id = source.match_id
   AND target.team_name = source.team_name
   AND target.player_name = source.player_name
WHEN MATCHED THEN
    UPDATE SET
        target.player_position = source.player_position
WHEN NOT MATCHED THEN
    INSERT (
        match_id,
        team_name,
        player_name,
        player_position
    )
    VALUES (
        source.match_id,
        source.team_name,
        source.player_name,
        source.player_position
    )

In [0]:
%sql
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT match_id) AS total_matchs,
    COUNT(DISTINCT team_name) AS total_teams,
    COUNT(DISTINCT player_position) AS total_positions
FROM
    workspace.futbol.silver_players
;

In [0]:
%sql
SELECT
*
FROM
workspace.futbol.silver_players
DESC
LIMIT 5;